# QMCPy Performance Optimizations Demo

Sou-Cheng T. Choi

Illinois Institute of Technology

Creation date: 8/14/2026

For reproducibility, this notebook was run with:
- Python 3.13.13, NumPy 2.5.0, SciPy 1.17.1, QMCPy 2.4, PyTorch (for the multitask kernel section)
- OS: macOS 15.6.1

This notebook benchmarks a set of small, targeted optimizations applied to QMCPy's internals: replacing generic `scipy.stats` distribution-object calls with the underlying `scipy.special` C-level functions, and replacing `np.einsum` calls (run with its default, non-BLAS-routed contraction path) with `@`/`matmul`.

**Why these are fast:**

- `scipy.stats.norm.ppf(x)` is a method on a generic `rv_continuous` distribution object. Even though it ultimately calls the same underlying Cephes routine, it first pays for loc/scale handling, support-bounds validation, and edge-case masking. `scipy.special.ndtri(x)` (and `ndtr(x)` for the forward CDF) call that routine directly.
- `np.einsum(subscripts, A, B)` is a general-purpose tensor-contraction *interpreter*. Unless called with `optimize=True`, it does not automatically recognize that a given contraction is really just a matrix multiply, so it doesn't route through BLAS's `GEMM` kernel the way `@`/`np.matmul` does. For contractions that *are* matrix multiplies in disguise, this can be a 10-50x difference.

Each section below defines a small "legacy" function using the old approach, times it against the equivalent QMCPy call (which now uses the fast approach), and reports the speedup. All outputs are verified numerically identical (up to floating-point rounding) before timing.

In [1]:
import time
import numpy as np
from scipy.stats import norm
from scipy.special import ndtri, ndtr

import qmcpy as qp
from qmcpy import DigitalNetB2, GeometricBrownianMotion, BrownianMotion, GaussianCopula
from qmcpy.kernel.multitask_kernel import KernelMultiTask
from qmcpy.kernel.common_kernels import KernelGaussian
from qmcpy.integrand.bayesian_lr_coeffs import BayesianLRCoeffs

rng = np.random.default_rng(7)

def bench(f, reps=10):
    f()  # warm up
    t0 = time.time()
    for _ in range(reps):
        result = f()
    return result, (time.time() - t0) / reps

results_table = []

def report(name, t_old, t_new, max_diff):
    speedup = t_old / t_new
    results_table.append((name, t_old, t_new, speedup, max_diff))
    print(f'{name}')
    print(f'  old: {t_old:.5f} s   new: {t_new:.5f} s   speedup: {speedup:.1f}x   max diff: {max_diff:.2e}')


## 1. Gaussian PCA Transform (`qmcpy/true_measure/gaussian.py`)

This is the code path used by default (`decomp_type="PCA"`) by `Gaussian`, `BrownianMotion`, and `GeometricBrownianMotion` -- so it's exercised any time `demos/GBM/gbm_demo.ipynb` samples GBM paths with a Sobol', Lattice, or Halton sampler. `A` below stands in for the cached PCA factor matrix.

In [2]:
n, d = 2**14, 252
x = rng.random((n, d))
A = rng.standard_normal((d, d))
mu = np.zeros(d)

def legacy_gaussian_transform():
    return mu + np.einsum('...ij,kj->...ik', norm.ppf(x), A)

def fast_gaussian_transform():
    out = ndtri(x) @ A.T
    out += mu
    return out

r_old, t_old = bench(legacy_gaussian_transform)
r_new, t_new = bench(fast_gaussian_transform)
report('Gaussian PCA transform (gaussian.py)', t_old, t_new, np.abs(r_old - r_new).max())


Gaussian PCA transform (gaussian.py)
  old: 0.24567 s   new: 0.03568 s   speedup: 6.9x   max diff: 1.21e-13


## 2. Brownian Bridge Transform (`qmcpy/true_measure/brownian_motion.py`)

Used whenever `decomp_type="BrownianBridge"` -- the construction showcased in `demos/brownian_bridge.ipynb`. We compare `BrownianMotion`'s actual bridge transform against a copy of the same code with `ndtri` swapped back for `norm.ppf`.

Note this section's speedup is more modest than the others: `_bridge_transform` itself still runs an O(d) sequential Python loop over dimensions (unrelated to this session's changes), which dominates total runtime here and dilutes the `ndtri` win. The einsum/ndtri techniques shown in this notebook do not address that loop.

In [3]:
d, n_paths = 256, 2**12
bm = BrownianMotion(DigitalNetB2(d, seed=7), decomp_type='BrownianBridge')
u = rng.random((n_paths, d))

def legacy_bridge_transform():
    z = norm.ppf(u)
    w = bm._bridge_transform(z)
    paths = bm.drift_time_vec_plus_init + np.sqrt(bm.diffusion) * w
    return paths[..., bm._output_order]

def fast_bridge_transform():
    return bm._transform(u)

r_old, t_old = bench(legacy_bridge_transform)
r_new, t_new = bench(fast_bridge_transform)
report('Brownian bridge transform (brownian_motion.py)', t_old, t_new, np.abs(r_old - r_new).max())


Brownian bridge transform (brownian_motion.py)
  old: 0.03263 s   new: 0.02254 s   speedup: 1.4x   max diff: 0.00e+00


## 3. Gaussian Copula Density (`qmcpy/true_measure/gaussian_copula.py`)

Used by `GaussianCopula._weight()`, showcased in `demos/copula_examples.ipynb`. This combines *both* techniques: `norm.ppf` -> `ndtri`, and a 3-operand quadratic-form `einsum` -> a matmul-then-reduce.

In [4]:
d, n = 40, 2**14
M = rng.standard_normal((d, d)); M = M @ M.T  # stand-in for corr_inv_minus_eye
u = rng.random((n, d))
logdet = 0.0

def legacy_copula_quad():
    z = norm.ppf(u)
    quad = np.einsum('...i,ij,...j->...', z, M, z)
    return -0.5 * logdet - 0.5 * quad

def fast_copula_quad():
    z = ndtri(u)
    quad = ((z @ M) * z).sum(-1)
    return -0.5 * logdet - 0.5 * quad

r_old, t_old = bench(legacy_copula_quad)
r_new, t_new = bench(fast_copula_quad)
report('Gaussian copula log-density (gaussian_copula.py)', t_old, t_new, np.abs(r_old - r_new).max())


Gaussian copula log-density (gaussian_copula.py)
  old: 0.03864 s   new: 0.00575 s   speedup: 6.7x   max diff: 7.50e-12


## 4. Multitask Kernel Matrix (`qmcpy/kernel/multitask_kernel.py`)

`KernelMultiTask.taskmat` builds a batched task-covariance matrix for multi-output/multi-fidelity Bayesian cubature. It previously used `einsum("...ij,...kj->...ik", ...)`; now it uses a batched `matmul`.

In [5]:
batch, num_tasks, rank = 2**10, 8, 8
factor = rng.standard_normal((batch, num_tasks, rank))

def legacy_taskmat():
    return np.einsum('...ij,...kj->...ik', factor, factor)

def fast_taskmat():
    return np.matmul(factor, np.swapaxes(factor, -1, -2))

r_old, t_old = bench(legacy_taskmat)
r_new, t_new = bench(fast_taskmat)
report('Multitask kernel taskmat (multitask_kernel.py)', t_old, t_new, np.abs(r_old - r_new).max())


Multitask kernel taskmat (multitask_kernel.py)
  old: 0.00038 s   new: 0.00009 s   speedup: 4.2x   max diff: 7.11e-15


## 5. Bayesian Logistic Regression Integrand (`qmcpy/integrand/bayesian_lr_coeffs.py`)

`BayesianLRCoeffs.g(x)` evaluates the log-likelihood integrand at each sampled coefficient vector; showcased in `demos/vectorized_qmc_bayes.ipynb`. The `einsum("...j,ij->...i", x, feature_array)` step is really `x @ feature_array.T`.

In [6]:
n, n_coeffs, n_obs = 2**14, 30, 5
x = rng.standard_normal((n, n_coeffs))
feature_array = rng.standard_normal((n_obs, n_coeffs))

def legacy_bayesian_lr():
    return np.einsum('...j,ij->...i', x, feature_array)

def fast_bayesian_lr():
    return x @ feature_array.T

r_old, t_old = bench(legacy_bayesian_lr)
r_new, t_new = bench(fast_bayesian_lr)
report('Bayesian LR feature projection (bayesian_lr_coeffs.py)', t_old, t_new, np.abs(r_old - r_new).max())


Bayesian LR feature projection (bayesian_lr_coeffs.py)
  old: 0.00035 s   new: 0.00024 s   speedup: 1.4x   max diff: 1.07e-14


## Summary

In [7]:
import pandas as pd
summary_df = pd.DataFrame(
    results_table,
    columns=['Component', 'Old (s)', 'New (s)', 'Speedup', 'Max Abs Diff'],
)
summary_df.round(6)


,Component,Old (s),New (s),Speedup,Max Abs Diff
0,Gaussian PCA transform (gaussian.py),0.245670,0.035680,6.885425,0.0
1,Brownian bridge transform (brownian_motion.py),0.032630,0.022536,1.447901,0.0
2,Gaussian copula log-density (gaussian_copula.py),0.038636,0.005753,6.716109,0.0
3,Multitask kernel taskmat (multitask_kernel.py),0.000382,0.000090,4.224215,0.0
4,Bayesian LR feature projection (bayesian_lr_co...,0.000347,0.000243,1.427996,0.0


**Where these speedups show up in existing demos**, next time each is re-run:

- `demos/GBM/gbm_demo.ipynb`, `demos/GBM/gbm_examples.ipynb` -- GeometricBrownianMotion and BrownianMotion path generation (Section 1 and 2 above)
- `demos/brownian_bridge.ipynb` -- BrownianMotion with `decomp_type="BrownianBridge"` (Section 2)
- `demos/copula_examples.ipynb`, `demos/product_measure.ipynb` -- GaussianCopula (Section 3)
- `demos/vectorized_qmc.ipynb`, `demos/vectorized_qmc_bayes.ipynb` -- BayesianLRCoeffs (Section 5)

No existing demo currently exercises `KernelMultiTask` (Section 4); it is used internally by multi-output/multi-fidelity Bayesian cubature stopping criteria.

## References

$[1]$ Harris, C.R., Millman, K.J., van der Walt, S.J. et al. (2020). Array programming with NumPy. *Nature* 585, 357-362.

$[2]$ Virtanen, P., Gommers, R., Oliphant, T.E. et al. (2020). SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python. *Nature Methods*, 17(3), 261-272.